# Семинар 5: Transfer Learning, Adversarial Attacks, Deep Network Visualization

## Вступление
Сегодня мы продолжим работать с изображениями при помощи нейросетей. Сперва узнаем, как потратить мало вычислительных ресурсов, чтобы при наличии модели для одной задачи, научиться решат новую задачу. Затем посмотрим на то, как можно за один шаг обучения сбивать нейросети с толку на некоторых примерах. В конце семинара будем визуализировать работу свёрточных сетей.

Этот ноутбук, в отличие от предыдущих, очень рекомендуется исполнять в Google Colab или похожих платформах, чтобы всю работало побыстрее.

### План семинара
1. Изучаем transfer learning на примере файнтюнинга ResNet
2. Изучаем adversarial attacks на примере Fast Gradient Sign Attack
3. Визуализируем слои свёрточной сети при помощи Shapley Additive Explanations

In [ ]:
import glob
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torchvision.datasets import ImageFolder
from torchvision.transforms import Compose, Normalize, Resize, ToTensor

%matplotlib inline
warnings.filterwarnings("ignore")

device = "cuda:0" if torch.cuda.is_available() else "cpu"

## 1. Изучаем transfer learning на примере файнтюнинга ResNet

В этой секции мы научимся очень быстро обучать нейросеть на сложную задачу классификации изображений. А именно, научим нейросеть отличать кошечек от собачек.

### Данные

Скачем датасет, удалим пару битых картинок и подготовим лоадеры.

In [ ]:
!wget https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip
!unzip -qq kagglecatsanddogs_5340.zip
!rm -rf PetImages/Cat/666.jpg PetImages/Dog/11702.jpg readme\[1\].txt CDLA-Permissive-2.0.pdf

In [ ]:
dataset = ImageFolder(
    "./PetImages",
    transform=Compose(
        [
            Resize((224, 224)),
            ToTensor(),
            Normalize((0.5, 0.5, 0.5), (1, 1, 1)),
        ]
    ),
)
train_set, val_set = torch.utils.data.random_split(
    dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))]
)

train_dataloader = torch.utils.data.DataLoader(train_set, batch_size=256, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_set, batch_size=256, shuffle=False)

Посмотрим, как выглядят картинки.

In [ ]:
file = np.random.choice(glob.glob("./PetImages/*/*.jpg"))
plt.axis("off")
plt.imshow(plt.imread(file));

### Fine-tuning

Кошки и собаки — это, конечно, хорошо, вот только обучение модели, которая будет хорошо работать на этом датасете может оказаться очень долгим...

Однако картинки, которые мы сегодня рассмотрим, оказываются очень похожими на картинки из огромного датасета ImageNet. Задача, которую мы сегодня рассмотрим, называется Transfer Learning. Знания мы действительно переносим — от сети, которая хорошо работает на одном датасете (ImageNet) к другим данным (к датасету Cats vs Dogs).

#### Загрузим уже обученную сеть

В библиотеке `torchvision` имплементировано не только большое множество моделей (всевозможные ResNet'ы, Inception, VGG, AlexNet, DenseNet, ResNext, WideResNet, MobileNet...), но и загружены чекпоинты обучения этих моделей на ImageNet. Однако для датасета Cats vs Dogs такая штука является роскошью...

In [ ]:
# !pip install pytorch_lightning torchmetrics torchvision

try:
    import accelerate, transformers, torchmetrics, torchvision  # noqa
except Exception:
    !pip -q install accelerate transformers torchmetrics torchvision

In [ ]:
from accelerate import Accelerator
from transformers import Trainer, TrainingArguments
from torchmetrics.functional import accuracy
from torchvision.models import ResNet18_Weights, resnet18

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model

В задаче transfer learning'a мы заменяем последний слой нейросети на линейный с двумя выходами, а веса остальных слоёв "замораживаем".

In [ ]:
class CatsVSDogsResnet(nn.Module):
    def __init__(self, pretrained: bool = False) -> None:
        super().__init__()
        self.pretrained = pretrained

        # <YOUR CODE HERE>

        # <YOUR CODE HERE>
        raise NotImplementedError('Реализуйте подготовку ResNet18 и заморозку backbone.')

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # <YOUR CODE HERE>
        # <YOUR CODE HERE>
        raise NotImplementedError('Реализуйте forward.')

    def train(self, mode: bool = True):
        super().train(mode)

        if self.pretrained:
            self.model.eval()

        return self


Инициализируем функции для обучения, которые уже кучу раз использовали.

Сначала обучим модель обычным циклом PyTorch, как в предыдущих семинарах. Затем повторим эксперимент с Accelerate и HF Trainer.

In [ ]:
from tqdm.auto import tqdm


def training_epoch(
    model,
    optimizer,
    criterion,
    train_loader,
    device,
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in tqdm(
        train_loader,
        desc="Training",
        leave=False,
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()
        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )


@torch.no_grad()
def validation_epoch(
    model,
    criterion,
    val_loader,
    device,
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in tqdm(
        val_loader,
        desc="Validation",
        leave=False,
    ):
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            logits.argmax(dim=1) == labels
        ).sum().item()
        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )

In [ ]:
def train(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    num_epochs,
    device,
    scheduler=None,
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_accuracy = training_epoch(
            model=model,
            optimizer=optimizer,
            criterion=criterion,
            train_loader=train_loader,
            device=device,
        )

        val_loss, val_accuracy = validation_epoch(
            model=model,
            criterion=criterion,
            val_loader=val_loader,
            device=device,
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        if scheduler is not None:
            scheduler.step()

        print(
            f"Epoch {epoch}/{num_epochs} | "
            f"train loss: {train_loss:.4f}, "
            f"accuracy: {train_accuracy:.4f} | "
            f"val loss: {val_loss:.4f}, "
            f"accuracy: {val_accuracy:.4f}"
        )

    return history

Запустим transfer learning!

Загрузим предобученную ResNet, заморозим backbone и обучим новый классификатор для кошек и собак.

In [ ]:
from accelerate.utils import set_seed

SEED = 42
EPOCHS = 1
LR = 1e-3

In [ ]:
set_seed(SEED)

cats_vs_dogs_pretrained_pytorch = CatsVSDogsResnet(
    pretrained=True
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer_pretrained_pytorch = ...  # YOUR CODE HERE

history_pretrained_pytorch = train(
    model=cats_vs_dogs_pretrained_pytorch,
    optimizer=optimizer_pretrained_pytorch,
    criterion=criterion,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    num_epochs=EPOCHS,
    device=device,
)

Уже после одной эпохи получается приемлемое качество! Давайте проверим, что получится, если учить ResNet с нуля.

Архитектура остаётся той же, но теперь веса backbone и классификатора инициализируются случайно и обучаются вместе. Используем те же данные, learning rate и число эпох.

In [ ]:
set_seed(SEED)

cats_vs_dogs_scratch_pytorch = CatsVSDogsResnet(
    pretrained=False
).to(device)

optimizer_scratch_pytorch = ...  # YOUR CODE HERE

history_scratch_pytorch = train(
    model=cats_vs_dogs_scratch_pytorch,
    optimizer=optimizer_scratch_pytorch,
    criterion=criterion,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    num_epochs=EPOCHS,
    device=device,
)

Как мы видим, на transfer learning'e нейросеть сходится очень быстро. Значительно быстрее, чем инициализированная с нуля. Можно с уверенностью говорить, что transfer learning — очень полезная техника.

Можно и через Accelerate или Trainer.

Посмотрим как это будет для Accelerate.

In [ ]:
def fit_accelerate(
    model,
    optimizer,
    criterion,
    train_loader,
    val_loader,
    num_epochs,
):
    accelerator = Accelerator(mixed_precision="no")

    model, optimizer, train_loader, val_loader = accelerator.prepare(
        model,
        optimizer,
        train_loader,
        val_loader,
    ) # prepare модели, оптимайзера и даталоадеров

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    for epoch in range(1, num_epochs + 1):
        # Обучение
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        for images, labels in tqdm(
            train_loader,
            desc="Training",
            leave=False,
            disable=not accelerator.is_local_main_process,
        ):
            # Подготовленный loader уже переносит батч на устройство.
            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = criterion(logits, labels)

            accelerator.backward(loss) # вместо loss.backward()
            optimizer.step()

            # Метрики по всем процессам без построения нового графа.
            with torch.no_grad():
                # gather_for_metrics — для сбора метрик
                # особенно важно при работе на нескольких устройствах
                metric_logits, metric_labels = (
                    accelerator.gather_for_metrics(
                        (logits.detach(), labels)
                    )
                )

                batch_size = metric_labels.size(0)
                batch_loss = criterion(
                    metric_logits,
                    metric_labels,
                )

                total_loss += batch_loss.item() * batch_size
                total_correct += (
                    metric_logits.argmax(dim=1) == metric_labels
                ).sum().item()
                total_samples += batch_size

        train_loss = total_loss / total_samples
        train_accuracy = total_correct / total_samples

        # Валидация
        model.eval()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in tqdm(
                val_loader,
                desc="Validation",
                leave=False,
                disable=not accelerator.is_local_main_process,
            ):
                logits = model(images)

                metric_logits, metric_labels = (
                    accelerator.gather_for_metrics(
                        (logits, labels)
                    )
                )

                batch_size = metric_labels.size(0)
                batch_loss = criterion(
                    metric_logits,
                    metric_labels,
                )

                total_loss += batch_loss.item() * batch_size
                total_correct += (
                    metric_logits.argmax(dim=1) == metric_labels
                ).sum().item()
                total_samples += batch_size

        val_loss = total_loss / total_samples
        val_accuracy = total_correct / total_samples

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        accelerator.print(
            f"Epoch {epoch}/{num_epochs} | "
            f"train loss: {train_loss:.4f}, "
            f"accuracy: {train_accuracy:.4f} | "
            f"val loss: {val_loss:.4f}, "
            f"accuracy: {val_accuracy:.4f}"
        )

    model = accelerator.unwrap_model(model)

    return model, history

In [ ]:
set_seed(SEED)

cats_vs_dogs_pretrained_accelerate = CatsVSDogsResnet(
    pretrained=True
) # уже без .to(device) — для этого есть prepare

optimizer_pretrained_accelerate = torch.optim.Adam(
    (
        parameter
        for parameter in cats_vs_dogs_pretrained_accelerate.parameters()
        if parameter.requires_grad
    ),
    lr=LR,
)

(
    cats_vs_dogs_pretrained_accelerate,
    history_pretrained_accelerate,
) = fit_accelerate(
    model=cats_vs_dogs_pretrained_accelerate,
    optimizer=optimizer_pretrained_accelerate,
    criterion=criterion,
    train_loader=train_dataloader,
    val_loader=val_dataloader,
    num_epochs=EPOCHS,
)

В Accelerate не так много всего меняется, потому что всё ещё нужно расписывать цикл обучения. Давайте посмотрим как это выглядит для Trainer.

Наша ResNet возвращает логиты, а Trainer ожидает немного другой интерфейс:
- батч — словарь с изображениями `pixel_values` и ответами `labels`;
- модель возвращает словарь с `logits` и, если переданы ответы, `loss`.

In [ ]:
class CatsVSDogsForTrainer(nn.Module):
    def __init__(self, net, criterion):
        super().__init__()
        self.net = net
        self.criterion = criterion

    def forward(self, pixel_values, labels=None):
        logits = self.net(pixel_values)

        output = {"logits": logits}

        if labels is not None:
            output["loss"] = self.criterion(logits, labels)

        return output


def collate_fn(examples):
    images, labels = zip(*examples)

    return {
        "pixel_values": torch.stack(images),
        "labels": torch.tensor(labels, dtype=torch.long),
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    return {
        "accuracy": float((predictions == labels).mean())
    }

In [ ]:
def fit_trainer(
    model,
    criterion,
    train_dataset,
    val_dataset,
    num_epochs,
    output_dir,
):
    wrapped_model = CatsVSDogsForTrainer(
        net=model,
        criterion=criterion,
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,

        # Используем размеры батчей исходных даталоадеров.
        per_device_train_batch_size=train_dataloader.batch_size,
        per_device_eval_batch_size=val_dataloader.batch_size,

        learning_rate=LR,
        weight_decay=0.0,
        lr_scheduler_type="constant",
        warmup_steps=0,
        max_grad_norm=0.0,
        gradient_accumulation_steps=1,

        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",

        remove_unused_columns=False,

        seed=SEED,
        data_seed=SEED,

        use_cpu=str(device) == "cpu",
        fp16=False,
        bf16=False,
    )

    trainer = Trainer(
        model=wrapped_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=collate_fn,
        compute_metrics=compute_metrics,

        optimizer_cls_and_kwargs=(
            torch.optim.Adam,
            {"lr": LR},
        ),
    )

    trainer.train()

    train_logs = [
        entry
        for entry in trainer.state.log_history
        if "loss" in entry
    ]

    val_logs = [
        entry
        for entry in trainer.state.log_history
        if "eval_loss" in entry
    ]

    history = {
        "train_loss": [
            entry["loss"]
            for entry in train_logs
        ],
        "val_loss": [
            entry["eval_loss"]
            for entry in val_logs
        ],
        "val_accuracy": [
            entry["eval_accuracy"]
            for entry in val_logs
        ],
    }

    trained_model = trainer.accelerator.unwrap_model(
        trainer.model
    ).net

    return trained_model, history

In [ ]:
set_seed(SEED)

cats_vs_dogs_pretrained_trainer = CatsVSDogsResnet(
    pretrained=True
)

(
    cats_vs_dogs_pretrained_trainer,
    history_pretrained_trainer,
) = fit_trainer(
    model=cats_vs_dogs_pretrained_trainer,
    criterion=criterion,
    train_dataset=train_set,
    val_dataset=val_set,
    num_epochs=EPOCHS,
    output_dir="./trainer_pretrained",
)

## 2. Изучаем adversarial attacks на примере Fast Gradient Sign Attack

Такая вещь, как атаки на нейросеть крайне важны для учёта при разработке. Существует много методов как их генерации, так и защиты от них. Мы рассмотрим сегодня базовые концепты, чтобы дать понимание происходящего.

Можем назвать adversarial атакой генерацию такого примера, который не отличим глазом от настоящего, но нейросеть будет ОЧЕНЬ уверена в том, что этот пример из другого класса. Сейчас мы попробуем сгенерировать такую собачку, что нейросеть будет уверена, что это котик.

Сегодня мы рассмотрим пример Fast Gradient Sign Attack (FGSM, почему там буква M в конце — чёрт его знает...). Идея очень простая. Оказывается, что если мы через обученную нейросеть посчитаем градиент по исходной картинке, посчитаем его знак и прибавим, умножив на маленькое число, модель подумает, что это картинка другого класса.

<img src="https://pytorch.org/tutorials/_images/fgsm_panda_image.png">

In [ ]:
def fgsm_attack(
    original_image: torch.Tensor, epsilon: float, data_grad: torch.Tensor
) -> torch.Tensor:
    # <YOUR CODE HERE>
    raise NotImplementedError("Реализуйте шаг FGSM.")


In [ ]:
attack_model = cats_vs_dogs_pretrained_pytorch.to(device)
# attack_model = cats_vs_dogs_pretrained_accelerate.to(device)
# attack_model = cats_vs_dogs_pretrained_trainer.to(device)

attack_model.eval()

for param in attack_model.parameters():
    param.requires_grad = True

In [ ]:
attack_model = cats_vs_dogs_pretrained_pytorch.to(device)

# Если хочется использовать модель, обученную через Accelerate:
# attack_model = cats_vs_dogs_pretrained_accelerate.to(device)

# Если хочется использовать модель, обученную через HF Trainer:
# attack_model = cats_vs_dogs_pretrained_trainer.to(device)

attack_model.eval()

while True:
    # Ищем кошку.
    idx = np.random.randint(0, len(train_set))
    if train_set[idx][1] != 0:
        continue

    cat_image = train_set[idx][0]

    # Разрешим вычисление градиента по картинке.
    cat_image.requires_grad = True

    pred = attack_model(cat_image[None].to(device))
    predicted_label = pred.argmax(1).item()

    # Берём только кошку, которую модель правильно распознала.
    if predicted_label == 0:
        break

confidence = pred.softmax(1)[0][predicted_label]
if predicted_label == 1:
    plt.title("Dog, confidence = %0.4f" % confidence.item())
else:
    plt.title("Cat, confidence =  %0.4f" % confidence.item())

plt.axis("off")
plt.imshow(cat_image.cpu().detach().numpy().transpose((1, 2, 0)) + 0.5);

In [ ]:
loss = nn.CrossEntropyLoss()(pred, torch.tensor(0).reshape((1,)).to(device))
loss.backward()

eps = 0.007

cat_image_perturbated = fgsm_attack(cat_image, eps, cat_image.grad)

# Используем ту же модель, которую выбрали в предыдущей ячейке:
# PyTorch, Accelerate или HF Trainer.
pred = attack_model(cat_image_perturbated[None].to(device))

predicted_label = pred.argmax(1).item()
confidence = pred.softmax(1)[0][predicted_label]

plt.axis("off")
if predicted_label == 1:
    plt.title("Dog, confidence = %0.4f" % confidence.item())
else:
    plt.title("Cat, confidence =  %0.4f" % confidence.item())
plt.imshow(cat_image_perturbated.cpu().detach().numpy().transpose((1, 2, 0)) + 0.5);

Самое интересное начинается тут. Вычислим градиент функции потерь по картинке и произведём атаку.

## 3. Визуализируем слои свёрточной сети при помощи Shapley Additive Explanations

<img src="https://raw.githubusercontent.com/slundberg/shap/master/docs/artwork/shap_header.svg" />

**SHAP (SHapley Additive exPlanations)** is a game theoretic approach to explain the output of any machine learning model. It connects optimal credit allocation with local explanations using the classic Shapley values from game theory and their related extensions (see papers for details and citations).

Представим, что предсказание - результат командной игры и нужно понять, какой вклад в какое место результат внес каждый игрок(фича).

Вклад каждой фичи измеряется так называемой **Shapley value** - понятием из **cooperative game theory**, описывающим вклад фичи в предсказание.

Например, вы снимаете однушку в Москве и хотите создать модель, которая предскажет стоимость выбранного вами варианта. Для заданной квартиры модель предсказала цену 50.000 р. Свойства квартиры, по которой была предсказана цена: в пределах мкада, есть кондиционер, с домашними крокодилами нельзя, рядом находится парк. Нужно найти **Shapley value** свойства "с крокодилами нельзя", чтобы понять, насколько большой вклад вносит именно этот признак.

**Shapley value** это среднее от возможных **marginal contributions**. Мы создаем все возможные наборы над множеством свойств, исключая интересующее ("с крокодилами нельзя"):


*   ни одного свойства
*   в пределах мкада
*   кондей
*   рядом парк
*   в пределах мкада, кондей
*   в пределах мкада, рядом парк
*   кондей, рядом парк
*   в пределах мкада, рядом парк, кондей


Пример для маржинального вклада:

**Marginal contribution** = *model_predict* (в пределах мкада, рядом парк, с крокодилами нельзя) -  *model_predict* (в пределах мкада, рядом парк)

$$
\Phi(v)_i=\sum_{K \ni i} \frac{(k-1) !(n-k) !}{n !}(v(K)-v(K \backslash i))
$$

Где: $n$ — количество игроков (в нашем случае это признаки модели). $v$ — предсказание модели на наборе признаков. $k$ — количество участников коалиции $K$.

Предсказание можно объяснить, предположив, что каждый признак является “игроком” в игре, где предсказание является выплатой. Значения Шепли — метод из теории коалиционных игр — подсказывает нам, как справедливо распределить “выплату” между признаками.

Документация по [cсылкe](https://shap.readthedocs.io/en/latest/).


In [ ]:
!pip install shap

In [ ]:
import json

import shap

### Изучаем слои ResNet на данных ImageNet

Трактовка предсказаний на основе оригинальной картинки проще, чем на основе слоёв, которые ближе всего к выходу сети, потому что последние менее интерпретируемы. Каждый уровень в deep CNN изучает фильтры возрастающей сложности. На первых слоях изучаются основные фильтры обнаружения объектов, такие как края и углы. Средние слои изучают фильтры, которые обнаруживают части объектов — что касается лиц, они могут научиться реагировать на глаза и носы. Последние слои имеют более высокие представления: они учатся распознавать объекты целиком, в различных формах и положениях.

Мы рассмотрим модель GradientExplainer, которая использует ожидаемые градиенты для оценки входов в разные части модели. В целом, они аппроксимируют значения SHAP. Будем брать 50 семплов для подсчёта ожидаемых градиентов и посмотрим на признаки первых слоёв ResNet.

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]


def normalize(image):
    if image.max() > 1:
        image /= 255
    image = (image - mean) / std
    # in addition, roll axes so that they suit pytorch
    return torch.tensor(image.swapaxes(-1, 1).swapaxes(2, 3)).float()

In [ ]:
# Load proper model (resnet50) and data (imagenet50) from Shap wrapper
model = resnet18(pretrained=True).eval()
X, y = shap.datasets.imagenet50()
X /= 255

to_explain = X[[1, 41]]

# load the ImageNet class names
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
fname = shap.datasets.cache(url)

with open(fname) as f:
    class_names = json.load(f)

#### Layer 1

In [ ]:
# create GradientExplainer object for convolution layer from Resnet Layer1
e = shap.GradientExplainer((model, model.layer1[0].conv2), normalize(X))
# receive shap_values
shap_values, indexes = e.shap_values(
    normalize(to_explain), ranked_outputs=4, nsamples=50
)

# get the names for the classes
index_names = np.vectorize(lambda x: class_names[str(x)][1])(indexes)

# plot the explanations
# SHAP результат: (изображения, каналы, высота, ширина, объясняемые выходы)
# нам надо для image_plot (изображения, высота, ширина, каналы)
shap_values = [
    np.moveaxis(shap_values[..., i], 1, -1)
    for i in range(shap_values.shape[-1])
]

shap.image_plot(shap_values, to_explain, index_names)

#### Layer 2

In [ ]:
e = shap.GradientExplainer((model, model.layer2[0].conv2), normalize(X))
shap_values, indexes = e.shap_values(
    normalize(to_explain), ranked_outputs=4, nsamples=50
)
index_names = np.vectorize(lambda x: class_names[str(x)][1])(indexes)
shap_values = [
    np.moveaxis(shap_values[..., i], 1, -1)
    for i in range(shap_values.shape[-1])
]
shap.image_plot(shap_values, to_explain, index_names)

#### Layer 4

In [ ]:
e = shap.GradientExplainer((model, model.layer4[0].conv2), normalize(X))
shap_values, indexes = e.shap_values(
    normalize(to_explain), ranked_outputs=4, nsamples=50
)
index_names = np.vectorize(lambda x: class_names[str(x)][1])(indexes)
shap_values = [
    np.moveaxis(shap_values[..., i], 1, -1)
    for i in range(shap_values.shape[-1])
]
shap.image_plot(shap_values, to_explain, index_names)